# Notebook 1 — Balanceamento de Carga em Computação Paralela

Este notebook foi planejado para a aula integrada de **balanceamento de carga, consistência de memória, OpenMP (modelo mental) e MPI/SPMD**.

## Objetivo
Demonstrar, na prática, que:

- paralelizar não basta;
- a forma de **distribuir o trabalho** afeta diretamente o tempo total;
- tarefas com custo desigual expõem gargalos;
- estratégias de particionamento influenciam o desempenho.

## Ideia central
Vamos simular um conjunto de tarefas com custos diferentes e comparar:

1. execução sequencial;
2. paralelização com divisão ingênua em blocos;
3. paralelização com distribuição mais equilibrada;
4. análise de tempo, speedup e ocupação dos workers.

> Mensagem-chave: **mais workers não garantem melhor desempenho se a carga estiver mal distribuída.**


In [ ]:
import math
import time
import statistics
from collections import defaultdict
from concurrent.futures import ProcessPoolExecutor, as_completed

import matplotlib.pyplot as plt


## 1. Gerando uma carga de trabalho desbalanceada

Em aplicações reais, nem todas as tarefas custam o mesmo.  
Algumas imagens podem exigir mais pré-processamento, certos lotes podem ser maiores, e alguns arquivos podem levar mais tempo para serem analisados.

Aqui, vamos criar tarefas artificiais com pesos diferentes.


In [ ]:
def gerar_tarefas():
    import random

    random.seed(42)

    tarefas = []

    # Muitas tarefas leves (simulando pré-processamento simples)
    tarefas += [random.randint(2, 10) for _ in range(150)]

    # Algumas tarefas médias
    tarefas += [random.randint(40, 55) for _ in range(30)]

    # Poucas tarefas MUITO pesadas espalhadas
    tarefas += [random.randint(100, 200) for _ in range(10)]

    # 🔥 Embaralhar apenas o final
    inicio_shuffle = 90
    sublista = tarefas[inicio_shuffle:]
    random.shuffle(sublista)
    tarefas[inicio_shuffle:] = sublista

    return tarefas

tarefas = gerar_tarefas()
len(tarefas), tarefas[:12]


In [ ]:
plt.figure(figsize=(12, 4))
plt.bar(range(len(tarefas)), tarefas)
plt.title("Distribuição dos pesos das tarefas")
plt.xlabel("Índice da tarefa")
plt.ylabel("Peso/custo relativo")
plt.show()


## 2. Função que simula trabalho computacional

Para que a comparação seja perceptível, cada tarefa executa um volume de operações proporcional ao seu peso.

Não estamos simulando I/O; queremos um cenário mais próximo de uma carga **CPU-bound**.


In [ ]:
def trabalho_pesado(peso: int) -> dict:
    inicio = time.perf_counter()

    # Carga artificial CPU-bound
    acumulador = 0.0
    repeticoes = peso * 120_000
    for i in range(1, repeticoes):
        acumulador += math.sqrt(i % 997) * math.sin(i % 360)

    fim = time.perf_counter()
    return {
        "peso": peso,
        "resultado": acumulador,
        "duracao": fim - inicio
    }


## 3. Baseline sequencial

Primeiro medimos a execução sequencial.  
Esse valor será nossa referência para calcular o **speedup**.


In [ ]:
def executar_sequencial(tarefas):
    inicio = time.perf_counter()
    resultados = [trabalho_pesado(t) for t in tarefas]
    fim = time.perf_counter()
    return {
        "resultados": resultados,
        "wall_time": fim - inicio
    }

baseline = executar_sequencial(tarefas)
baseline["wall_time"]


## 4. Estratégia 1 — Particionamento em blocos (ingênuo)

Aqui dividimos a lista em blocos contíguos.  
Essa estratégia é simples, mas pode concentrar tarefas pesadas em poucos workers.


In [ ]:
def dividir_em_blocos(tarefas, n_workers):
    n = len(tarefas)
    tamanho_bloco = math.ceil(n / n_workers)
    blocos = []
    for i in range(0, n, tamanho_bloco):
        blocos.append(tarefas[i:i+tamanho_bloco])
    while len(blocos) < n_workers:
        blocos.append([])
    return blocos


In [ ]:
def worker_bloco(worker_id, bloco):
    inicio = time.perf_counter()
    resultados = [trabalho_pesado(t) for t in bloco]
    fim = time.perf_counter()
    return {
        "worker_id": worker_id,
        "n_tarefas": len(bloco),
        "carga_total": sum(bloco),
        "wall_time": fim - inicio,
        "resultados": resultados
    }


In [ ]:
def executar_blocos(tarefas, n_workers=4):
    blocos = dividir_em_blocos(tarefas, n_workers)
    inicio = time.perf_counter()
    saidas = []

    with ProcessPoolExecutor(max_workers=n_workers) as ex:
        futures = [ex.submit(worker_bloco, wid, bloco) for wid, bloco in enumerate(blocos)]
        for fut in as_completed(futures):
            saidas.append(fut.result())

    fim = time.perf_counter()
    saidas = sorted(saidas, key=lambda x: x["worker_id"])
    return {
        "particoes": blocos,
        "saidas": saidas,
        "wall_time": fim - inicio
    }

exec_blocos = executar_blocos(tarefas, n_workers=4)
exec_blocos["wall_time"]


## 5. Estratégia 2 — Particionamento guloso balanceado

Agora distribuímos cada nova tarefa para o worker que está com a menor carga acumulada até aquele momento.

Essa heurística simples já costuma melhorar bastante o equilíbrio.


In [ ]:
def dividir_balanceado_guloso(tarefas, n_workers):
    particoes = [[] for _ in range(n_workers)]
    cargas = [0] * n_workers

    for tarefa in sorted(tarefas, reverse=True):
        idx = min(range(n_workers), key=lambda i: cargas[i])
        particoes[idx].append(tarefa)
        cargas[idx] += tarefa

    return particoes


In [ ]:
def executar_balanceado(tarefas, n_workers=4):
    particoes = dividir_balanceado_guloso(tarefas, n_workers)
    inicio = time.perf_counter()
    saidas = []

    with ProcessPoolExecutor(max_workers=n_workers) as ex:
        futures = [ex.submit(worker_bloco, wid, parte) for wid, parte in enumerate(particoes)]
        for fut in as_completed(futures):
            saidas.append(fut.result())

    fim = time.perf_counter()
    saidas = sorted(saidas, key=lambda x: x["worker_id"])
    return {
        "particoes": particoes,
        "saidas": saidas,
        "wall_time": fim - inicio
    }

exec_balanceado = executar_balanceado(tarefas, n_workers=4)
exec_balanceado["wall_time"]


## 6. Comparando as partições

Repare como a carga total por worker muda de uma estratégia para outra.


In [ ]:
def resumo_particoes(execucao):
    return [
        {
            "worker": s["worker_id"],
            "n_tarefas": s["n_tarefas"],
            "carga_total": s["carga_total"],
            "tempo_worker": s["wall_time"]
        }
        for s in execucao["saidas"]
    ]

resumo_blocos = resumo_particoes(exec_blocos)
resumo_balanceado = resumo_particoes(exec_balanceado)

resumo_blocos, resumo_balanceado


In [ ]:
workers_b = [r["worker"] for r in resumo_blocos]
cargas_b = [r["carga_total"] for r in resumo_blocos]
tempos_b = [r["tempo_worker"] for r in resumo_blocos]

workers_g = [r["worker"] for r in resumo_balanceado]
cargas_g = [r["carga_total"] for r in resumo_balanceado]
tempos_g = [r["tempo_worker"] for r in resumo_balanceado]

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].bar(workers_b, cargas_b)
axes[0].set_title("Carga por worker — blocos")
axes[0].set_xlabel("Worker")
axes[0].set_ylabel("Carga total")

axes[1].bar(workers_g, cargas_g)
axes[1].set_title("Carga por worker — guloso balanceado")
axes[1].set_xlabel("Worker")
axes[1].set_ylabel("Carga total")

plt.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].bar(workers_b, tempos_b)
axes[0].set_title("Tempo por worker — blocos")
axes[0].set_xlabel("Worker")
axes[0].set_ylabel("Tempo (s)")

axes[1].bar(workers_g, tempos_g)
axes[1].set_title("Tempo por worker — guloso balanceado")
axes[1].set_xlabel("Worker")
axes[1].set_ylabel("Tempo (s)")

plt.tight_layout()
plt.show()


## 7. Speedup e eficiência

Vamos calcular métricas simples:

- **Speedup** = tempo sequencial / tempo paralelo
- **Eficiência** = speedup / número de workers


In [ ]:
def metricas(tempo_seq, tempo_par, n_workers):
    speedup = tempo_seq / tempo_par
    eficiencia = speedup / n_workers
    return speedup, eficiencia

speedup_blocos, eficiencia_blocos = metricas(baseline["wall_time"], exec_blocos["wall_time"], 4)
speedup_balanceado, eficiencia_balanceado = metricas(baseline["wall_time"], exec_balanceado["wall_time"], 4)

print(f"Sequencial: {baseline['wall_time']:.4f} s")
print(f"Blocos:     {exec_blocos['wall_time']:.4f} s | speedup={speedup_blocos:.3f} | eficiência={eficiencia_blocos:.3f}")
print(f"Balanceado: {exec_balanceado['wall_time']:.4f} s | speedup={speedup_balanceado:.3f} | eficiência={eficiencia_balanceado:.3f}")


In [ ]:
labels = ["Sequencial", "Blocos", "Balanceado"]
tempos = [baseline["wall_time"], exec_blocos["wall_time"], exec_balanceado["wall_time"]]

plt.figure(figsize=(8, 4))
plt.bar(labels, tempos)
plt.title("Comparação de tempo total")
plt.ylabel("Tempo (s)")
plt.show()


## 8. Interpretação dos resultados

Pontos para observar:

1. O tempo total paralelo não depende da média dos workers, mas do **worker mais lento**.
2. Se uma estratégia concentra carga demais em poucos processos, o paralelismo perde eficiência.
3. O overhead de criação e coordenação existe; por isso, nem sempre o speedup é linear.
4. Balanceamento é um fator central em HPC, MPI, Spark e pipelines de IA.


In [ ]:
tempo_critico_blocos = max(r["tempo_worker"] for r in resumo_blocos)
tempo_critico_balanceado = max(r["tempo_worker"] for r in resumo_balanceado)

print(f"Worker crítico (blocos):     {tempo_critico_blocos:.4f} s")
print(f"Worker crítico (balanceado): {tempo_critico_balanceado:.4f} s")


## 9. Relação com OpenMP e MPI

### OpenMP (modelo mental)
Quando usamos uma região paralela com `parallel for`, a forma como as iterações são distribuídas entre threads influencia o tempo total.  
Escalonamentos diferentes (`static`, `dynamic`, `guided`) existem justamente para atacar esse problema.

### MPI / SPMD
Em MPI, cada processo executa o mesmo programa sobre uma fatia dos dados.  
Se o particionamento for ruim, alguns ranks terminam cedo e ficam ociosos, esperando os mais lentos.

> Em ambos os casos, o problema central é o mesmo: **quem recebe qual trabalho?**


## 10. Exercícios propostos

### Exercício 1
Altere a lista de tarefas para deixá-la ainda mais desbalanceada.  
Depois, repita os testes e compare os resultados.

### Exercício 2
Teste com `n_workers = 2`, `4`, `8`, `10` e `12`.  
O ganho cresce linearmente? Por quê?

### Exercício 3
Implemente uma estratégia cíclica (round-robin) e compare com:
- blocos;
- guloso balanceado.


In [ ]:
# Estratégia 3 — Particionamento Cíclico (Round-Robin)

# Faça aqui sua implementação

## 11. Síntese final

Este notebook mostrou que:

- paralelismo sem boa distribuição de carga gera ociosidade;
- o desempenho depende da estratégia de particionamento;
- o gargalo costuma estar no **último worker a terminar**;
- balanceamento de carga é uma ponte direta entre teoria e prática em HPC.

Na próxima etapa, faz sentido avançar para um notebook que conecte isso com **modelo mental de OpenMP** ou com **SPMD/MPI prático**.
